# Signature Forgery Detector — Training Walkthrough
This notebook demonstrates dataset exploration, Siamese network pair generation, model initialization, and training/evaluation.

In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt

sys.path.append("..")
from src.dataset import create_sample_dataset_if_missing, load_signature_paths, SignaturePairDataset
from src.model import SiameseNet, EmbeddingNet
from src.loss import ContrastiveLoss

dataset_dir = create_sample_dataset_if_missing("../data")
print(f"Loaded dataset directory: {dataset_dir}")

## 1. Inspect Sample Pairs

In [ ]:
genuine_dict, forged_dict = load_signature_paths(dataset_dir)
person_ids = list(genuine_dict.keys())
dataset = SignaturePairDataset(dataset_dir, person_ids, is_train=False, pairs_per_person=10)

img1, img2, label = dataset[0]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(img1.squeeze(), cmap='gray')
ax1.set_title("Signature 1")
ax2.imshow(img2.squeeze(), cmap='gray')
ax2.set_title(f"Signature 2 (Label={label.item()})")
plt.show()

## 2. Test Model Forward Pass

In [ ]:
model = SiameseNet(EmbeddingNet())
emb1, emb2 = model(img1.unsqueeze(0), img2.unsqueeze(0))
dist = torch.nn.functional.pairwise_distance(emb1, emb2)
print(f"Embedding shape: {emb1.shape}")
print(f"Initial distance score: {dist.item():.4f}")